# Threshold tuning for the TF-IDF + Logistic Regression baseline

This notebook picks a better cutoff **per category**, on contracts the model never trained on and
that are also separate from the contracts we grade on, then compares before vs. after.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED_DIR = Path(os.getcwd()).parent / "data" / "processed"
if not PROCESSED_DIR.exists():
    raise FileNotFoundError(f"{PROCESSED_DIR} not found -- run main_pipeline.ipynb first to create it")

chunks_df = pd.read_parquet(PROCESSED_DIR / "chunks_df.parquet")
chunk_labels_df = pd.read_parquet(PROCESSED_DIR / "chunk_labels_df.parquet")

# one row per chunk, one 0/1 column per category (same as main_pipeline.ipynb's baseline cell)
label_matrix = chunk_labels_df.pivot_table(
    index="chunk_id", columns="category", values="label", fill_value=0
)
categories = label_matrix.columns.tolist()
assert len(categories) == 41, "project rules say predict all 41 CUAD categories, not a reduced set"

baseline_df = chunks_df.set_index("chunk_id")[["contract_id", "chunk_text"]].join(label_matrix)
baseline_df = baseline_df.dropna(subset=["chunk_text"])

n_pos = int(baseline_df[categories].values.sum())
n_pairs = baseline_df[categories].size
print(f"{len(baseline_df)} chunks from {baseline_df['contract_id'].nunique()} contracts, "
      f"{len(categories)} categories")
print(f"{n_pos} of {n_pairs} (chunk, category) pairs are yes ({n_pos / n_pairs * 100:.2f}%)")

8031 chunks from 408 contracts, 41 categories
8233 of 329271 (chunk, category) pairs are yes (2.50%)


In [2]:
from sklearn.model_selection import GroupShuffleSplit

# split 1 -- unchanged from main_pipeline.ipynb: practice pool vs. validation, by contract
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
pool_idx, val_idx = next(splitter.split(baseline_df, groups=baseline_df["contract_id"]))
pool_df, val_df = baseline_df.iloc[pool_idx], baseline_df.iloc[val_idx]

# split 2 -- carve the cutoff-picking set out of the practice pool, by contract
tune_splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
train_idx, tune_idx = next(tune_splitter.split(pool_df, groups=pool_df["contract_id"]))
train_df, tune_df = pool_df.iloc[train_idx], pool_df.iloc[tune_idx]

# sanity checks: no contract may appear in more than one set
train_c, tune_c, val_c = (set(d["contract_id"]) for d in (train_df, tune_df, val_df))
assert not (train_c & tune_c), "a contract is in both train_df and tune_df"
assert not (train_c & val_c) and not (tune_c & val_c), "a contract leaked into val_df"

for name, d in [("train", train_df), ("tune", tune_df), ("validation", val_df)]:
    print(f"{name:>10}: {len(d):5d} chunks from {d['contract_id'].nunique():3d} contracts")

     train:  5101 chunks from 260 contracts
      tune:  1216 chunks from  66 contracts
validation:  1714 chunks from  82 contracts


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# word scores -- same settings as main_pipeline.ipynb; vocabulary learned from train_df only
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english")
X_train = vectorizer.fit_transform(train_df["chunk_text"])
X_tune = vectorizer.transform(tune_df["chunk_text"])
X_val = vectorizer.transform(val_df["chunk_text"])
y_train, y_tune, y_val = (d[categories].values.astype(int) for d in (train_df, tune_df, val_df))


def get_balanced_row_mask(y, rng):
    """Copied from main_pipeline.ipynb: keep every yes, randomly keep about as many nos."""
    pos_mask = y == 1
    neg_mask = ~pos_mask
    n_pos, n_neg = pos_mask.sum(), neg_mask.sum()
    if n_pos == 0 or n_neg == 0:
        return np.ones_like(y, dtype=bool)
    neg_keep_frac = min(1.0, n_pos / n_neg)
    neg_keep = neg_mask & (rng.random(len(y)) < neg_keep_frac)
    return pos_mask | neg_keep


# one model per category, trained exactly like the baseline -- but keep the confidence
# (0-1, column 1 of predict_proba = chance of "yes") instead of predict()'s built-in 50% cutoff
rng = np.random.default_rng(0)
tune_scores = np.zeros(y_tune.shape)
val_scores = np.zeros(y_val.shape)
for i, cat in enumerate(categories):
    keep = get_balanced_row_mask(y_train[:, i], rng)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train[keep], y_train[keep, i])
    tune_scores[:, i] = clf.predict_proba(X_tune)[:, 1]
    val_scores[:, i] = clf.predict_proba(X_val)[:, 1]

print(f"confidence scores: tune {tune_scores.shape}, validation {val_scores.shape} (chunks x categories)")
print(f"share of tune scores above 50%: {(tune_scores > 0.5).mean() * 100:.1f}% "
      f"vs. real yes rate {y_tune.mean() * 100:.1f}%")

confidence scores: tune (1216, 41), validation (1714, 41) (chunks x categories)
share of tune scores above 50%: 18.6% vs. real yes rate 2.6%


In [4]:
from sklearn.metrics import fbeta_score

CANDIDATE_THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.01), 2)  # 5%, 6%, ... 95%
DEFAULT_THRESHOLD = 0.5
# how many times more "finding real clauses" (recall) counts than "being right when saying yes"
# (precision) when grading a cutoff. 1 = balanced (plain F1). We use 3: a missed clause is never
# seen by the reviewer, while a wrong yes only costs them a quick skim.
RECALL_WEIGHT = 3


def pick_threshold(y_true, scores, candidate_thresholds, recall_weight):
    """Return the cutoff with the best recall-weighted score (F-beta) for ONE category.
    Ties go to the lowest cutoff (argmax returns the first best), which keeps more real clauses."""
    grades = [fbeta_score(y_true, scores >= t, beta=recall_weight, zero_division=0)
              for t in candidate_thresholds]
    return float(candidate_thresholds[int(np.argmax(grades))])


def pick_all_thresholds(recall_weight):
    """One cutoff per category, picked on the tune contracts only -- validation is not looked at here."""
    picked = []
    for i in range(len(categories)):
        if y_tune[:, i].sum() == 0:
            picked.append(DEFAULT_THRESHOLD)  # no real examples in tune to judge a cutoff by
        else:
            picked.append(pick_threshold(y_tune[:, i], tune_scores[:, i], CANDIDATE_THRESHOLDS, recall_weight))
    return np.array(picked)


thresholds_balanced = pick_all_thresholds(recall_weight=1)        # for comparison only
thresholds = pick_all_thresholds(recall_weight=RECALL_WEIGHT)     # the ones we keep

threshold_df = pd.DataFrame({
    "n_yes_in_tune": y_tune.sum(axis=0),
    "threshold": thresholds,
}, index=categories).sort_values("threshold")
print(threshold_df.to_string())
print(f"\nmedian cutoff: {threshold_df['threshold'].median():.2f} (was {DEFAULT_THRESHOLD} for every category)")

                                    n_yes_in_tune  threshold
Source Code Escrow                              5       0.35
No-Solicit Of Customers                        10       0.39
License Grant                                  75       0.45
Price Restrictions                              4       0.46
Minimum Commitment                             43       0.47
Non-Disparagement                               9       0.47
Exclusivity                                    36       0.48
Irrevocable Or Perpetual License               29       0.49
Revenue/Profit Sharing                         38       0.50
Expiration Date                                76       0.50
Rofr/Rofo/Rofn                                 13       0.51
Non-Compete                                    21       0.52
Warranty Duration                              24       0.52
Agreement Date                                 64       0.52
Affiliate License-Licensor                      2       0.52
Non-Transferable License

In [5]:
from sklearn.metrics import f1_score, precision_score, recall_score

# turn validation confidences into yes/no three ways
pred_before = (val_scores >= DEFAULT_THRESHOLD).astype(int)      # 50% for every category
pred_balanced = (val_scores >= thresholds_balanced).astype(int)  # cutoffs picked for best F1
pred_after = (val_scores >= thresholds).astype(int)              # cutoffs picked with recall counted 3x


def summarize(y_true, y_pred):
    """Overall precision/recall/F1: 'micro' pools every (chunk, category) pair, 'macro' averages the 41 categories."""
    out = {}
    for avg in ("micro", "macro"):
        out[f"{avg} precision"] = precision_score(y_true, y_pred, average=avg, zero_division=0)
        out[f"{avg} recall"] = recall_score(y_true, y_pred, average=avg, zero_division=0)
        out[f"{avg} F1"] = f1_score(y_true, y_pred, average=avg, zero_division=0)
    out["wrong yeses"] = int(((y_pred == 1) & (y_true == 0)).sum())
    out["correct yeses"] = int(((y_pred == 1) & (y_true == 1)).sum())
    return out


summary_df = pd.DataFrame({
    "before (50%)": summarize(y_val, pred_before),
    "balanced (F1)": summarize(y_val, pred_balanced),
    f"recall x{RECALL_WEIGHT} (kept)": summarize(y_val, pred_after),
})
print(f"validation: {int(y_val.sum())} real yeses out of {y_val.size} (chunk, category) pairs\n")
print(summary_df.round(3).to_string())

per_category_df = pd.DataFrame({
    "n_yes_in_val": y_val.sum(axis=0),
    "threshold": thresholds,
    "recall_before": recall_score(y_val, pred_before, average=None, zero_division=0),
    "recall_after": recall_score(y_val, pred_after, average=None, zero_division=0),
    "precision_before": precision_score(y_val, pred_before, average=None, zero_division=0),
    "precision_after": precision_score(y_val, pred_after, average=None, zero_division=0),
}, index=categories)
print(f"\nper-category results on validation, before (50%) vs. after (recall x{RECALL_WEIGHT} cutoffs):\n")
print(per_category_df.sort_values("n_yes_in_val", ascending=False).round(2).to_string())

validation: 1664 real yeses out of 70274 (chunk, category) pairs

                 before (50%)  balanced (F1)  recall x3 (kept)
micro precision         0.116          0.388             0.229
micro recall            0.812          0.463             0.723
micro F1                0.203          0.422             0.348
macro precision         0.160          0.401             0.234
macro recall            0.718          0.381             0.639
macro F1                0.246          0.346             0.314
wrong yeses         10265.000       1212.000          4054.000
correct yeses        1351.000        770.000          1203.000

per-category results on validation, before (50%) vs. after (recall x3 cutoffs):

                                    n_yes_in_val  threshold  recall_before  recall_after  precision_before  precision_after
License Grant                                107       0.45           0.90          0.93              0.48             0.34
Parties                              